# Detección y medición del ciclo diurno en un huracán

Los datos utilizados en este Cuaderno JupyterLab fueron generados en el cuaderno
[05_Calcular_perfil_radial.ipynb](./05_Calcular_perfil_radial.ipynb).
Corresponden a los perfiles radiales del campo de temperatura de brillo
centrado en la tormenta y reproyectados usando una **Proyección Acimutal
Equidistante**.

La finalidad de este cuaderno es detectar el ciclo diurno de un ciclón tropical
o subtropical y calcular su período a partir de la evolución temporal de la
diferencia de perfiles radiales.

## 1. Preliminares

### 1.1 Configuración del laboratorio

Puesta a punto del entorno, carga de configuraciones, e inicialización del proyecto.

In [ ]:
from goesdl.experimental.utilities import initialize_project, reload_project

# Setup the environment, load settings, and initialise the project
settings = initialize_project("../event.yaml", "../settings.yaml")

### 1.2 Carga del inventario de archivos

Carga del inventario de archivos de datos.

In [ ]:
from goesdl.experimental.utilities import load_inventory

dataset_paths, dataset_times = load_inventory(settings)

### 1.3 Parámetros calculados y dependientes de los datos

Calcula los parámetros de configuración que dependen de los datos a analizar.

In [ ]:
from goesdl.experimental.utilities import get_computed_parameters

parameters = get_computed_parameters(settings, dataset_paths)

## 2. Ejecución del algoritmo

### 2.1 Paso 1: Generación de series de tiempo

1. Carga cronológica de los perfiles radiales, $P(r)_{t}$ de $T_{bb}$, para cada instante $t$ a analizar
2. Diferencia de perfiles radiales $D(r)_{t} = P(r)_{t} - P(r)_{t+\delta t}$, con una separación temporal, $\delta t$
3. Creación de la serie de tiempo $S(t)_{r} = D(r)_{t}$, para los valores de $r$ dados arriba

In [ ]:
from goesdl.experimental.algorithm import run_algorithm_w

# Create the time series holder
bt_raw_timeseries = run_algorithm_w(settings, parameters, dataset_paths)

### 2.2 Paso 2: Tratamiento de datos faltantes

#### 2.2.1 Eliminación de datos faltantes en los extremos

In [ ]:
from goesdl.experimental.utilities import trim_timeseries

bt_timeseries = trim_timeseries(bt_raw_timeseries, parameters)

#### 2.2.2: Imputación de datos faltantes

In [ ]:
from goesdl.experimental.utilities import fill_timeseries

bt_filled_timeseries, bt_gap_indices = fill_timeseries(bt_timeseries, settings)

### 2.3 Paso 3: Submuestreo

In [ ]:
from goesdl.experimental.utilities import subsample_timeseries

bt_subsampled_timeseries = subsample_timeseries(bt_filled_timeseries, settings, parameters)

### 2.4 Paso 4: Eliminación de tendencias

In [ ]:
from goesdl.experimental.utilities import detrend_timeseries

bt_detrended_timeseries, bt_tendency_components = detrend_timeseries(bt_subsampled_timeseries)

### 2.5 Paso 5: Computo de series de tiempo promedio

In [ ]:
from goesdl.experimental.utilities import calculate_mean_timeseries

bt_mean_timeseries = calculate_mean_timeseries(bt_subsampled_timeseries, bt_detrended_timeseries, settings, parameters)

print(len(bt_mean_timeseries))
for key, value in bt_mean_timeseries.items():
    print(f"{key} : {len(value)}")

### 2.6 Paso 6: Filtración de banda de interés

In [ ]:
from goesdl.experimental.utilities import filter_timeseries

bt_filtered_timeseries, bt_filtered_mean_timeseries = filter_timeseries(bt_detrended_timeseries, bt_mean_timeseries, settings, parameters)

### 2.7 Paso 7: Análisis espectral (Fourier)

In [ ]:
from goesdl.experimental.utilities import analyze_spectra, find_diurnal_cycle, get_dominant_cycle

analysers, average_analysers = analyze_spectra(bt_detrended_timeseries, bt_mean_timeseries, settings, parameters)

diurnal_cycle, mean_diurnal_cycle = find_diurnal_cycle(analysers, average_analysers)
dominant_cycle, mean_dominant_cycle = get_dominant_cycle(analysers, average_analysers)

## 3. Presentación de resultados

### 3.1 Visualización de las series de tiempo capturadas y puntos imputados

In [ ]:
from goesdl.experimental.report import visualizar_capturas

visualizar_capturas(bt_timeseries, bt_filled_timeseries, bt_mean_timeseries, bt_gap_indices, settings, parameters)

### 3.2 Visualización de series sin tendencia

Para realizar un análisis espectral correcto, debemos eliminar toda tendencia que pueda estar afectando a las series de tiempo. Estas tendencias pueden ser interpretadas erróneamente como oscilaciones de muy baja frecuencia de gran amplitud. En los ciclones, las tendencias pueden estar generadas por características del ciclo de vida del fenómeno.

Para este estudio, asumimos una tendencia lineal y la determinamos haciendo una regresión lineal a cada serie $S(t)_r$. La línea resultante, $y(t)_r = mt + b$, se substrae de la serie resultando una serie de tiempo sin tendencia lineal, $\hat S(t)_r = S(t)_r - y(t)_r$. El término constante $b$ es la media de la serie de tiempo.

La primera imagen muestra todas las series en una escala unificada. La segunda imagen superpone los promedios incoherentes original y sin tendencia, para dimensionar el grado de tendencia de las series originales. La última imagen muestra los promedios coherente e incoherente sin tendencia.

Los promedios incoherentes se calculan, simplemente, promediando todas las series de tiempo capturadas: $\bar S(t) = \frac{1}{N} \sum_{r=r_\text{m{\'i}n}}^{r_\text{m\'ax}} S(t)_r$, e igualmente con las series sin tendencia.

Se procede análogamente para obtener el promedio coherente, poniendo antes todas las series en fase para reducir la cancelación debido a la propagación de fenómeno.

Los promedios representarían el comportamiento medio del fenómeno a lo largo de los radios estudiados.

In [ ]:
from goesdl.experimental.report import visualizar_normalizados

visualizar_normalizados(bt_detrended_timeseries, bt_mean_timeseries, settings, parameters)

### 3.3 Visualización de series filtradas (opcional)

Permiten ver la contribución de cada serie de tiempo a la frecuencia central, o ciclo, que se está midiendo. Cuanto mayor la amplitud, más intenso o evidente el ciclo en la región medida.

El promedio coherente, obtenido promediando todas las series de tiempo alineadas en fase para reducir la cancelación debido a la propagación de fenómeno, muestra indicios tanto del fortalecimiento como de la extinción del fenómeno a medida transcurre el tiempo.

El promedio incoherente permite ver la cancelación de fase que evidencia la propagación radial del fenómeno con una rapidez finita. Esta rapidez puede estimarse midiendo la separacion entre picos de series de radio consecutivas y evidenciar si la propagación es uniforme o no a lo largo del radio. En principio, pareciera que la rapidez de propagaión aumenta con el radio.

In [ ]:
from goesdl.experimental.report import visualizar_filtrados

visualizar_filtrados(bt_filtered_timeseries, bt_filtered_mean_timeseries, settings, parameters)

### 3.4 Visualización de periodogramas

Un ***periodograma*** o *espectro de potencia*, muestra la distribución de la **densidad espectral de potencia** de una serie de tiempo en función de la **frecuencia**.

La **densidad espectral de potencia** de una señal es una función matemática que nos informa de cómo está distribuida la potencia de dicha señal sobre las distintas frecuencias de las que está formada.

> Aunque la densidad espectral no es exactamente lo mismo que el espectro de una señal, a veces ambos términos se usan indistintamente, lo cual, en rigor, es incorrecto.

#### 3.4.1 Cómo interpretar los resultados

**Picos significativos**: Cuando la potencia espectral supera la línea del percentil, $\rho$, correspondiente, sugiere que esa frecuencia contiene señal real, no solo ruido, con nivel de confianza estadística $\rho = 100 \cdot (1 - \alpha)\%$ o, dicho de otra manera, con una significancia igual $100 \cdot (\alpha)\%$. $\rho$ es una medida de cuán confiados queremos estar.

El **valor** $p$, o $p$-valor, es una estadístico que nos muestra la probabilidad de haber obtenido el resultado que hemos logrado suponiendo que la hipótesis nula $H_0$ es cierta. Valores altos de $p$ no permiten rechazar $H_0$, mientras que valores bajos de $p$ sí permiten rechazar $H_0$.

En una prueba estadística, se rechaza la hipótesis nula $H_0$ si el valor $p$ asociado al resultado observado es igual o menor que un nivel de significación $\alpha$ establecido arbitrariamente, convencionalmente $0.05$ o $0.01$. En otras palabras, si el resultado obtenido es más inusual que el rango esperado de resultados dada una hipótesis nula $H_0$ cierta y el nivel de significación $\alpha$ elegido, es decir si $p$ es menor que $\alpha$, podemos decir que tenemos un resultado estadísticamente significativo que permite rechazar $H_0$.

Es importante recalcar que un contraste de hipótesis no permite **aceptar** una hipótesis; simplemente la rechaza o no la rechaza, es decir que la tacha de verosímil (lo que no significa obligatoriamente que sea cierta, simplemente que es más probable de serlo) o inverosímil.

> *La significación estadística de un resultado no implica que el resultado también tenga relevancia en el mundo real. Por ejemplo, un efecto estadísticamente significativo puede ser demasiado pequeño para ser interesante.*

##### Evaluación práctica:

* Picos por encima del percentil $95\%$ son estadísticamente significativos
* Picos entre percentiles $90\text{--}95\%$ son marginalmente significativos
* Picos por debajo del $90\%$ pueden ser ruido

Picos alrededor y por debajo de la línea nula

* **No significa**: "Definitivamente es ruido", "No hay información útil ahí"
* **Sí significa**: "Comportamiento consistente con variabilidad natural del ruido", "No hay evidencia de señal determinística fuerte"

Un pico por debajo de la línea nula tiene **alta probabilidad de ser una fluctuación aleatoria** del proceso de ruido, pero esta probabilidad no es abrumadoramente alta como en el caso de picos muy por encima de percentiles altos. Es una **región de "ruido típico"** donde no se pueden hacer afirmaciones fuertes en ninguna dirección.

##### Ruido Rojo (Red Noise)

El ruido rojo representa un proceso autoregresivo de primer orden AR(1), caracterizado por:

* Correlación positiva entre valores consecutivos
* Mayor potencia en frecuencias bajas que decae hacia altas frecuencias
* Los umbrales de ruido rojo son generalmente más altos que los de ruido blanco
* Es común en sistemas meteorológicos, climáticos y geofísicos debido a la inercia natural
* El ruido rojo es más realista para datos naturales con memoria temporal

##### Interpretación de percentiles

* La línea nula asume ruido rojo (comportamiento típico, sin correlación temporal), se denomina así porque representa la hipótesis nula, $H_0$
* Solo el $5\%$ del espectro de ruido rojo excederá la línea de percentil $95\%$ por casualidad, (probablemente señal real)
* Solo el $1\%$ excederá la línea de percentil $99\%$ aleatoriamente, (muy probablemente señal real)
* El intervalo de confianza del $100 \cdot (\alpha)\%$ es toda la región que se encuentra por encima de la línea de percentil $100\cdot(1 - \alpha)\%$

Esta metodología es especialmente importante en meteorología, climatología, oceanografía y geofísica, donde distinguir señales reales de variabilidad natural es fundamental para identificar ciclos, tendencias o periodicidades genuinas en los datos.

#### 3.4.2 Prueba de hipótesis

En el contexto de pruebas de significancia espectral se trata de una prueba de cola derecha unilateral.

* $H_0(f)$: El pico de potencia de densidad espectral, observado en la frecuencia $f$, proviene únicamente del proceso de ruido estocástico de fondo
* $H_1(f)$: El pico de potencia de densidad espectral observado contiene señal determinística real en la frecuencia $f$.

Rechazar $H_0(f)$ implica que existe una componente de señal determinística en la frecuencia $f$.

##### Test de Hipótesis Formal

* **Estadístico de prueba**: $T(f) = S_\text{obs}(f) / S_\text{nulo}(f)$
* **Distribución nula**: $T(f) \sim F(2, \infty)$ bajo $H_0$
* **Decisión**: Rechazar $H_0$ si $T(f) > F_{1 - \alpha}(2, \infty)$

donde $S_\star(f)$ es la potencia de la densidad espectral. En análisis espectral, la formulación correcta de $H_0$ debe ser específica por frecuencia $f$, no global.

##### Razones Metodológicas

1. **Independencia Estadística**: Cada frecuencia constituye una prueba de hipótesis independiente
2. **Distribución Espectral del Ruido**: La función del modelo nulo AR(1) varía con la frecuencia, por lo que $H_0$ debe evaluarse punto a punto
3. **Control de Error Tipo I**: Una formulación global crearía problemas de *comparaciones múltiples*:
    * Si evaluamos $N$ frecuencias con $\alpha = 0.05$ cada una
    * Probabilidad de al menos un falso positivo $\approx 1 - (1 - 0.05)^N$

##### Formulación Matemática Precisa

Para cada bandeja espectral $f_i$:

* $H_0(f_i)$: $S_\text{obs}(f_i) \sim S_\text{nulo}(f_i) \cdot \chi^2(2) / 2$
* $H_1(f_i)$: $S_\text{obs}(f_i) = S_\text{se\~nal}(f_i) + S_\text{nulo}(f_i) \cdot \chi^2(2) / 2$

donde $S_\text{se\~nal}(f_i) > 0$ representa la componente determinística.

#### 3.4.3 $p$-valor (Probabilidad de error si declaro significativo)

> Curva de $p$-valores: ¿Qué tan improbable es cada pico si fuera solo ruido?

##### Cómo explicarlo a otras personas

###### Lenguaje simple:

*"Este gráfico muestra qué tan sorprendente es cada pico del espectro. Valores bajos (cerca del fondo) significan 'muy sorprendente si fuera solo ruido', valores altos (cerca del tope) significan 'normal, probablemente ruido'."*

###### Analogía efectiva:

"Imagina que estás escuchando una orquesta tocando. El gráfico de $p$-valores te dice, para cada nota musical, qué tan probable es que sea solo el ruido de fondo del auditorio versus una nota real de los instrumentos. Valores muy bajos = definitivamente música real."

##### Guía de interpretación rápida del gráfico de $p$-valores

###### Para Audiencias Técnicas:

"Los $p$-valores cuantifican la probabilidad de que cada pico espectral sea una fluctuación aleatoria. Valores $< 0.05$ sugieren señales reales con $95\%$ de confianza."

###### Para Audiencias Generales:

"Este gráfico identifica qué partes de la señal son 'música real' versus 'ruido de fondo'. Las líneas que tocan el fondo del gráfico representan señales genuinas."

##### Puntos clave para presentaciones:

1. Escala logarítmica es esencial: Diferencias entre 0.001 y 0.01 son enormes
2. Distribución uniforme = solo ruido: Si $p$-valores están esparcidos uniformemente
3. Valles profundos = señales: Concentraciones de $p$-valores muy bajos
4. Contexto físico importa: Un $p = 0.04$ puede ser ruido, un $p = 10^{-6}$ definitivamente no

##### Frases Útiles para Explicar:

* *"Mientras más bajo el valle, más confiamos en que hay señal real"*
* *"La línea roja es nuestro umbral de confianza del $\,95\%$"*
* *"Si fuera solo ruido, esperaríamos ver una línea plana y ondulada"*

El gráfico de $p$-valores es tu "detector de señales cuantitativo" - te dice exactamente qué tan seguros puedes estar de cada componente espectral.

#### 3.4.4 Conclusión

La línea nula representa el **comportamiento típico del azar**, conceptualmente es el **centro de la distribución del azar**. Es la referencia central contra la cual medimos qué tan improbable es observar desviaciones hacia arriba, las líneas de percentiles.

Al afirmar que **un pico que sobrepasa el percentil $100\cdot(1 - \alpha)\%$ proviene de un proceso natural (señal real)**, existe una probabilidad de $100 \cdot (\alpha)\%$ de estar cometiendo un error, o también, la probabilidad de $100 \cdot (\alpha)\%$ de que lo que se esté afirmando sea falso. Pero la afirmación es **estadísticamente correcta**.

Esta es la interpretación clásica y correcta del $p$-valor en análisis espectral.

> **Y recuerden amigos**: *cuanto más valle, mejor...*, vaamonooo!


In [ ]:
from goesdl.experimental.report import crear_espectrogramas

# settings = reload_project("../event.yaml", "../settings.yaml", False)

crear_espectrogramas(analysers, average_analysers, settings, parameters)

### 3.5 Visualización de los ciclos dominantes en cada serie de tiempo

In [ ]:
from numpy import all as npall
from goesdl.experimental.utilities import combine_tick_labels, get_date_markers, get_time_ticks

ext_analysers = analysers + [average_analysers["coherent_mean_timeseries"]]
ext_detrended_timeseries = bt_detrended_timeseries + [bt_mean_timeseries["coherent_mean_timeseries"]]
ext_filtered_timeseries = bt_filtered_timeseries + [bt_filtered_mean_timeseries["coherent_mean_timeseries"]]
ext_dominant_cycle = dominant_cycle + [mean_dominant_cycle["coherent_mean_timeseries"]]
ext_diurnal_cycle = diurnal_cycle + [mean_diurnal_cycle["coherent_mean_timeseries"]]

amp_factor = 3

filter_frequency = settings["algorithm"]["filter_frequency"]
title_right = (f"(fs = {samples_per_day} muestras/d, dt={delta_hr:0.1f}h, fc = {filter_frequency} c/d)", "right")

dalpha = [(0 if npall(dcycle == 0) else 0.8) for dcycle in ext_diurnal_cycle]
talpha = [(0 if npall(dcycle == tcycle) else 1) for dcycle, tcycle in zip(ext_diurnal_cycle, ext_dominant_cycle)]

dlable = [(None if npall(dcycle == 0) else f"Ciclo diurno {amp_factor}×") for dcycle in ext_diurnal_cycle]
tlabel = [(None if npall(dcycle == tcycle) else f"Ciclo dominante {amp_factor}×") for dcycle, tcycle in zip(ext_diurnal_cycle, ext_dominant_cycle)]

sav_suptitle = settings["plotting"]["series"]["suptitle"]
settings["plotting"]["series"]["suptitle"] = None

sav_height = settings["plotting"]["series"]["height"]
settings["plotting"]["series"]["height"] = [ 4.5 ]

title_inset = [f"r = {radius_km:.0f}-km" for radius_km in radii_km]
title_inset.append("(promedio coherente)")

tick_position, tick_label, time_hours = get_time_ticks(settings, parameters)
mark_position, mark_label = get_date_markers(settings, parameters)
tick_label = combine_tick_labels(tick_position, tick_label, mark_position, mark_label)

xmarkers = [{"x":pos, "color":"black", "linestyle":"--", "alpha":0.7} for pos in mark_position]

for i, inset in enumerate(title_inset):
    dominant_index = ext_analysers[i].peak_indices[0]
    dominant_frequency = ext_analysers[i].frequencies[dominant_index]
    dominant_period = 1 / dominant_frequency
    title_center = [f"Serie de tiempo {inset}", "center"]
    title_right = (f"(f = {24*dominant_frequency:.2f} c/d, T = {dominant_period:.2f} h/c)", "right")
    timeseries = [
        [
            ext_detrended_timeseries[i],
            ext_filtered_timeseries[i],
            amp_factor*ext_dominant_cycle[i],
            amp_factor*ext_diurnal_cycle[i] 
        ],
    ]
    params = [    
        {
            "title": [title_center, title_right],
            "label": ["Serie original", "Serie filtrada", tlabel[i], dlable[i]],
            "xarray": 24*times_days,
            "xlabel": "Fecha  [h]",
            "ylabel": ylabel,
            "suptitle": None,
            "alpha": [0.6, 0.6, talpha[i], dalpha[i]],
            "linestyle": ["-", "-.", ":", "--"],
            "xmarkers": xmarkers,
            "xticks": tick_position,
            "xticklabels": {"labels": tick_label, "rotation": 45, "ha": "right"},
            "xlim": (0, time_hours)
        },
    ]

    plot_timeseries(timeseries, settings, params, "series")

settings["plotting"]["series"]["height"] = sav_height
settings["plotting"]["series"]["suptitle"] = sav_suptitle

In [ ]:
# In[ ]:
# Controles interactivos para los parámetros del algoritmo y visualización
# Definir valores iniciales y rangos
initial_diff_hours = 6.0
initial_radius_min = 100
initial_radius_step = 100
initial_central_mask = 0.0
initial_window_size = 100.0
initial_freq_sampling_rate = 2048 # 2 * 1024
initial_series_resolution = 1
initial_invert_difference = False
initial_window_function = "hann"
initial_central_frequency = 1.0
initial_band_width = 0.3
initial_display_images = False
initial_display_metadata = False
initial_verbose = False
initial_reported_periods = 5
initial_regenerate_analysis = False

# Crear widgets
difference_hours_slider = widgets.FloatSlider(
    value=initial_diff_hours, min=0.0, max=24.0, step=0.5,
    description='Diff Offset (Hrs):', continuous_update=False,
    orientation='horizontal', readout=True, readout_format='.1f'
)
radius_min_slider = widgets.IntSlider(
    value=initial_radius_min, min=0, max=500, step=10,
    description='Min Radius (km):', continuous_update=False
)
radius_step_slider = widgets.IntSlider(
    value=initial_radius_step, min=10, max=200, step=10,
    description='Radius Step (km):', continuous_update=False
)
central_mask_slider = widgets.FloatSlider(
    value=initial_central_mask, min=0.0, max=1.0, step=0.05,
    description='Central Mask (%):', continuous_update=False,
    readout_format='.2f'
)
windows_size_slider = widgets.FloatSlider(
    value=initial_window_size, min=0.0, max=100.0, step=5.0,
    description='Window Size (%):', continuous_update=False,
    readout_format='.1f'
)
frequency_sampling_rate_text = widgets.Dropdown(
    options=[(f'{2**i} (2^{i})', 2**i) for i in range(10, 13)] + [('2*1024', 2*1024)], # Opciones comunes para FFT size
    value=initial_freq_sampling_rate,
    description='FFT Size:',
    continuous_update=False
)
series_resolution_slider = widgets.IntSlider(
    value=initial_series_resolution, min=1, max=5, step=1,
    description='Series Res (f/hr):', continuous_update=False
)
invert_difference_checkbox = widgets.Checkbox(
    value=initial_invert_difference,
    description='Invert Diff P[i+n]-P[i]',
    disabled=False
)
window_function_dropdown = widgets.Dropdown(
    options=['bartlett', 'blackman', 'boxcar', 'hamming', 'hann'],
    value=initial_window_function,
    description='Window Func:',
)
central_frequency_slider = widgets.FloatSlider(
    value=initial_central_frequency, min=0.0, max=3.0, step=0.1,
    description='Central Freq (cyc/day):', continuous_update=False,
    readout_format='.1f'
)
band_width_slider = widgets.FloatSlider(
    value=initial_band_width, min=0.0, max=1.0, step=0.05,
    description='Band Width (cyc/day):', continuous_update=False,
    readout_format='.2f'
)

# Opciones de visualización
display_images_checkbox = widgets.Checkbox(
    value=initial_display_images,
    description='Mostrar Imágenes',
    disabled=False
)
display_metadata_checkbox = widgets.Checkbox(
    value=initial_display_metadata,
    description='Mostrar Metadatos',
    disabled=False
)
verbose_checkbox = widgets.Checkbox(
    value=initial_verbose,
    description='Modo Detallado',
    disabled=False
)
reported_periods_slider = widgets.IntSlider(
    value=initial_reported_periods, min=1, max=10, step=1,
    description='Periodos a Reportar:', continuous_update=False
)
regenerate_analysis_checkbox = widgets.Checkbox(
    value=initial_regenerate_analysis,
    description='Forzar Regeneración de Análisis',
    disabled=False
)

# Agrupar widgets en un Output para capturar los valores
output = widgets.Output()

def update_params(
    diff_hours, radius_min, radius_step, central_mask, window_size, freq_sampling_rate,
    series_res, invert_diff, window_func, central_freq, band_width,
    display_imgs, display_meta, verbose_mode, reported_per, regenerate_an
):
    global ALGORITHM_PARAMS, DISPLAY_OPTIONS

    ALGORITHM_PARAMS = {
        "DIFFERENCE_HOURS": diff_hours,
        "RADIUS_MIN": radius_min,
        "RADIUS_STEP": radius_step,
        "CENTRAL_MASK": central_mask,
        "WINDOWS_SIZE": window_size,
        "FREQUENCY_SAMPLING_RATE": freq_sampling_rate,
        "SERIES_RESOLUTION": series_res,
        "INVERT_DIFFERENCE": invert_diff,
        "WINDOW_FUNCTION": window_func,
        "CENTRAL_FREQUENCY": central_freq,
        "BAND_WIDTH": band_width,
    }

    DISPLAY_OPTIONS = {
        "DISPLAY_IMAGES": display_imgs,
        "DISPLAY_METADATA": display_meta,
        "VERBOSE": verbose_mode,
        "REPORTED_PERIODS": reported_per,
        "REGENERATE_ANALYSIS": regenerate_an,
    }

    with output:
        output.clear_output(wait=True)
        print("Parámetros del Algoritmo Actuales:")
        for k, v in ALGORITHM_PARAMS.items():
            print(f"  {k}: {v}")
        print("\nOpciones de Visualización Actuales:")
        for k, v in DISPLAY_OPTIONS.items():
            print(f"  {k}: {v}")

# Conectar los widgets a la función de actualización
interactive_widget = widgets.interactive_output(
    update_params,
    {
        'diff_hours': difference_hours_slider,
        'radius_min': radius_min_slider,
        'radius_step': radius_step_slider,
        'central_mask': central_mask_slider,
        'window_size': windows_size_slider,
        'freq_sampling_rate': frequency_sampling_rate_text,
        'series_res': series_resolution_slider,
        'invert_diff': invert_difference_checkbox,
        'window_func': window_function_dropdown,
        'central_freq': central_frequency_slider,
        'band_width': band_width_slider,
        'display_imgs': display_images_checkbox,
        'display_meta': display_metadata_checkbox,
        'verbose_mode': verbose_checkbox,
        'reported_per': reported_periods_slider,
        'regenerate_an': regenerate_analysis_checkbox,
    }
)

# Mostrar los widgets y el output
display(
    widgets.VBox([
        widgets.HTML("<h3>Parámetros del Algoritmo</h3>"),
        widgets.HBox([
            widgets.VBox([difference_hours_slider, radius_min_slider, radius_step_slider, central_mask_slider, windows_size_slider]),
            widgets.VBox([frequency_sampling_rate_text, series_resolution_slider, invert_difference_checkbox, window_function_dropdown]),
            widgets.VBox([central_frequency_slider, band_width_slider])
        ]),
        widgets.HTML("<h3>Opciones de Visualización y Control</h3>"),
        widgets.HBox([
            widgets.VBox([display_images_checkbox, display_metadata_checkbox, verbose_checkbox]),
            widgets.VBox([reported_periods_slider, regenerate_analysis_checkbox])
        ]),
        output
    ])
)

# Después de ejecutar esta celda, ALGORITHM_PARAMS y DISPLAY_OPTIONS estarán disponibles
# con los valores seleccionados por los widgets.